# Test Predict — загрузка PRD-модели из MLflow

**Цель:** загрузить финальную модель по тегу `stage=PRD` из MLflow Model Registry и сделать предикт на одной строке из датасета.

**Модель:** `GeoATM-LightGBM-PRD` (LightGBM Optuna-100)  
**MLflow:** http://localhost:5050  
**MinIO:** http://localhost:9000

## 1. Импорты и подключение к MLflow

In [12]:
import os
import numpy as np
import pandas as pd
import mlflow
import mlflow.pyfunc
from pathlib import Path

mlflow.set_tracking_uri('http://localhost:5050')

os.environ['AWS_ACCESS_KEY_ID']        = 'admin'
os.environ['AWS_SECRET_ACCESS_KEY']    = 'password'
os.environ['MLFLOW_S3_ENDPOINT_URL']   = 'http://localhost:9000'

print('Tracking URI:', mlflow.get_tracking_uri())

Tracking URI: http://localhost:5050


## 2. Найти последнюю PRD-версию модели в реестре

In [13]:
MODEL_NAME = 'GeoATM-LightGBM-PRD'

client = mlflow.tracking.MlflowClient()

# Берём последнюю версию модели из реестра
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest = sorted(versions, key=lambda v: int(v.version))[-1]

print(f'Модель    : {latest.name}')
print(f'Версия    : {latest.version}')
print(f'Run ID    : {latest.run_id}')
print(f'Source    : {latest.source}')

# Проверяем тег stage=PRD на run
run = client.get_run(latest.run_id)
stage_tag = run.data.tags.get('stage', 'не задан')
print(f'Tag stage : {stage_tag}')
assert stage_tag == 'PRD', f'Ожидался тег PRD, получили: {stage_tag}'

Модель    : GeoATM-LightGBM-PRD
Версия    : 1
Run ID    : 014c6c70a4204ab98ec10163fa324cf5
Source    : models:/m-c282c18e3ae141069721db74efaf1c0d
Tag stage : PRD


## 3. Загрузить модель

In [14]:
model_uri = f'models:/{MODEL_NAME}/{latest.version}'
print(f'Загружаем: {model_uri}')

model = mlflow.pyfunc.load_model(model_uri)
print('Модель загружена:', type(model))

Загружаем: models:/GeoATM-LightGBM-PRD/1


Модель загружена: <class 'mlflow.pyfunc.PyFuncModel'>


## 4. Подготовка данных

In [15]:
DATA_PATH = Path('../data/train_data_v2.csv')
df = pd.read_csv(DATA_PATH)
print(f'Датасет: {df.shape}')

BINARY_FEATURES = [
    'is_24_7', 'contactless_tech', 'qr_codes', 'usd_available', 'eur_available',
    'cash_in', 'cash_out', 'cashless_pay', 'account_statement', 'access_for_disabled',
    'transfer_p2p', 'transfer_a2a', 'loan_payments',
    'is_federal_city', 'is_federal_district_capital', 'is_city_center', 'has_subway_nearby',
]
DISTANCE_FEATURES = [
    'nearest_malls_dist_m', 'nearest_supermarkets_dist_m',
    'nearest_pharmacies_hospitals_dist_m', 'nearest_cafes_dist_m',
    'nearest_restaurants_dist_m', 'nearest_public_transport_dist_m',
    'nearest_parking_dist_m', 'nearest_education_dist_m', 'nearest_subway_dist_m',
    'nearest_post_offices_dist_m', 'nearest_offices_dist_m',
    'nearest_shops_food_small_dist_m', 'nearest_fitness_sport_dist_m',
    'nearest_hotels_hostels_dist_m', 'land_use_dist_m', 'city_center_dist_m',
    'nearest_residential_landuse_dist_m',
]
COUNT_FEATURES = [
    'count_malls_300m', 'count_supermarkets_300m', 'count_pharmacies_hospitals_300m',
    'count_banks_atms_300m', 'count_cafes_300m', 'count_restaurants_300m',
    'count_public_transport_300m', 'count_parking_300m', 'count_education_300m',
    'count_post_offices_300m', 'count_offices_300m', 'count_payment_terminals_300m',
    'count_money_transfer_300m', 'count_shops_food_small_300m', 'count_hypermarkets_300m',
    'count_markets_300m', 'count_fitness_sport_300m', 'count_hotels_hostels_300m',
    'count_railway_stations_300m', 'count_residential_buildings_300m',
    'count_residential_landuse_300m', 'count_fuel_300m',
    'count_highway_pedestrian_300m', 'count_landuse_mix_300m', 'count_footway_100m_100m',
]
ECONOMIC_FEATURES = [
    'avg_salary_oct_2025_rub', 'grp_per_capita_2023_rub',
    'avg_income_q3_2025_rub', 'population_density_per_km2',
]
CATEGORICAL_FEATURES = ['federal_district', 'city_size_category', 'land_use_type', 'atm_group']

ALL_FEATURES = [
    f for f in
    BINARY_FEATURES + DISTANCE_FEATURES + COUNT_FEATURES + ECONOMIC_FEATURES + CATEGORICAL_FEATURES
    if f in df.columns
]

def preprocess(data):
    X = data[ALL_FEATURES].copy()
    for col in BINARY_FEATURES:
        if col in X.columns and X[col].dtype == bool:
            X[col] = X[col].astype(int)
    for col in DISTANCE_FEATURES:
        if col in X.columns:
            X[col] = np.log1p(X[col].clip(lower=0))
    for col in CATEGORICAL_FEATURES:
        if col in X.columns:
            X[col] = X[col].fillna('unknown').astype('category')
    return X

print(f'Признаков: {len(ALL_FEATURES)}')

Датасет: (6200, 134)
Признаков: 67


## 5. Предикт на 10 случайных строках

In [16]:
sample = df.sample(10, random_state=42).reset_index(drop=True)
X_sample = preprocess(sample)
y_true = sample['target'].values
y_pred = model.predict(X_sample)

results = pd.DataFrame({
    'y_true':     y_true.round(6),
    'y_pred':     y_pred.round(6),
    'abs_error':  np.abs(y_true - y_pred).round(6),
})
display(results)

,y_true,y_pred,abs_error
0,-0.064220,-0.060317,0.003903
1,0.183003,0.179355,0.003648
2,0.116897,0.168503,0.051606
3,-0.060742,-0.081947,0.021205
4,0.013472,-0.018713,0.032185
5,-0.035929,-0.025517,0.010412
6,-0.129155,-0.059659,0.069496
7,-0.051421,-0.061552,0.010131
8,0.059294,0.047955,0.011340
9,0.154754,0.116145,0.038609
